<!-- LABDOC_AUTO -->
### 1. Carga del cat?logo y distribuci?n inicial

- **Teor?a:** Antes de modelar, se valida la poblaci?n etiquetada y el desbalance de clases; esto define si usaremos m?tricas macro y particiones estratificadas.
- **Llega:** `OGLE4-GSEP-full/list.dat`, le?do como cat?logo de ancho fijo desde el directorio descomprimido.
- **Sale:** `df`, conteos por `type`/`subtype` y figuras base en `figuras/`.
- **Insights:** La distribuci?n no es uniforme; por eso las evaluaciones posteriores usan estratificaci?n y m?tricas que no premian s?lo la clase mayoritaria.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)

from pathlib import Path



data_dir = Path("OGLE4-GSEP-full")
out_dir = Path("figuras")
out_dir.mkdir(exist_ok=True)
list_path = data_dir / "list.dat"

if not data_dir.exists():
    raise FileNotFoundError(f"No encontre el directorio descomprimido en:\n{data_dir.resolve()}")

if not list_path.exists():
    raise FileNotFoundError(f"No encontre list.dat en:\n{list_path.resolve()}")

# 1) Leer list.dat directamente desde el directorio descomprimido
# Columnas segun el README del catalogo OGLE
colspecs = [
    (0, 15),    # OGLE-IV ID
    (17, 28),   # RA J2000
    (29, 40),   # Dec J2000
    (41, 46),   # Variability type
    (47, 52),   # Subtype
    (54, 60),   # I mean mag
    (61, 67),   # V mean mag
    (68, 80),   # Period
    (81, 86),   # I amplitude
    (87, 99),   # Secondary period
]

cols = [
    "id", "ra", "dec", "type", "subtype",
    "I_mean", "V_mean", "period", "I_amp", "period2"
]

df = pd.read_fwf(list_path, colspecs=colspecs, names=cols, header=None)

# 2) Limpieza basica
for c in ["id", "ra", "dec", "type", "subtype"]:
    df[c] = df[c].astype(str).str.strip()
    df.loc[df[c].isin(["", "nan", "None"]), c] = np.nan

for c in ["I_mean", "V_mean", "period", "I_amp", "period2"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Clase principal y clase detallada
df["class_primary"] = df["type"]
df["class_full"] = np.where(
    df["subtype"].notna(),
    df["type"] + "_" + df["subtype"],
    df["type"]
)

# 3) Conteos
primary_counts = df["class_primary"].value_counts()
full_counts = df["class_full"].value_counts()

print("Numero total de objetos:", len(df))
print("\nClases principales:")
print(primary_counts)
print("\nClases detalladas, incluyendo subtipo:")
print(full_counts)

# 4) Histograma / barras de clases principales
plt.figure(figsize=(10, 5))
primary_counts.plot(kind="bar", color=SECONDARY, edgecolor=PRIMARY, linewidth=1.2)
plt.title("Distribucion de clases principales - OGLE-IV GSEP", color=INK)
plt.xlabel("Clase principal", color=INK)
plt.ylabel("Numero de objetos", color=INK)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

primary_png = out_dir / "ogle_histograma_clases_principales.png"
plt.savefig(primary_png, dpi=300, bbox_inches="tight")
plt.show()

# 5) Histograma / barras de todas las clases detalladas
plt.figure(figsize=(14, 6))
full_counts.plot(kind="bar", color=[PLOT_PALETTE[i % len(PLOT_PALETTE)] for i in range(len(full_counts))], edgecolor=PRIMARY, linewidth=0.8)
plt.title("Distribucion de clases detalladas - OGLE-IV GSEP", color=INK)
plt.xlabel("Clase detallada: type + subtype", color=INK)
plt.ylabel("Numero de objetos", color=INK)
plt.xticks(rotation=70, ha="right")
plt.tight_layout()

full_png = out_dir / "ogle_histograma_clases_detalladas.png"
plt.savefig(full_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nFiguras guardadas en:")
print(primary_png)
print(full_png)


<!-- LABDOC_AUTO -->
### 2. Agrupaci?n supervisada de clases

- **Teor?a:** Se reduce la taxonom?a fina de OGLE a familias f?sicas m?s estables: pulsantes, binarias, variables de largo per?odo y otros.
- **Llega:** `df` con la columna original `type`.
- **Sale:** `class_grouped` y una visualizaci?n de clases agrupadas con la paleta del laboratorio.
- **Insights:** `OTHER` se conserva para exploraci?n, pero se excluye del entrenamiento final porque mezcla objetos heterog?neos y degrada la interpretaci?n supervisada.


In [ ]:
# ============================================================
# Agrupar clases similares
# ============================================================

def agrupar_clases_ogle(tipo):
    """
    Agrupa clases del atlas OGLE en categorías más generales.
    """

    if tipo in ["DCEP", "T2CEP", "ACEP", "RRLYR", "DSCT" ]:
        return "PULS"

    elif tipo in ["ECL", "ELL"]:
        return "BINARY"

    elif tipo == "LPV":
        return "LPV"

    elif tipo == "OTHER":
        return "OTHER"

    else:
        return "UNKNOWN"


df["class_grouped"] = df["type"].apply(agrupar_clases_ogle)

# Conteo de clases agrupadas
grouped_counts = df["class_grouped"].value_counts()

print("Distribución de clases agrupadas:")
print(grouped_counts)



# ============================================================
# Histograma más estético de clases agrupadas
# ============================================================

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"
PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]

plt.figure(figsize=(10, 5))

bars = plt.bar(
    grouped_counts.index,
    grouped_counts.values,
    color=SECONDARY,
    edgecolor=PRIMARY,
    linewidth=1.5
)

plt.title("Distribución de clases agrupadas - OGLE-IV GSEP", fontsize=14, color=INK)
plt.xlabel("Clase agrupada", fontsize=12, color=INK)
plt.ylabel("Número de objetos", fontsize=12, color=INK)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.25)

# Escribir el número encima de cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=10,
        color=DARK
    )

plt.tight_layout()

grouped_png = out_dir / "ogle_histograma_clases_agrupadas.png"
plt.savefig(grouped_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nFigura guardada en:")
print(grouped_png.resolve())


<!-- LABDOC_AUTO -->
### 3. Composici?n interna de las clases agrupadas

- **Teor?a:** Un histograma apilado permite auditar si cada clase agrupada combina subtipos coherentes o si oculta mezclas problem?ticas.
- **Llega:** `class_grouped` y `type`.
- **Sale:** Tabla cruzada y figura apilada de composici?n interna en `figuras/`.
- **Insights:** `PULS` agrupa varias subfamilias pulsantes, `BINARY` combina eclipsantes/elipsoidales y `LPV` domina en cantidad; esto justifica usar `macro_f1`.


In [ ]:
# ============================================================
# Histograma apilado: clases agrupadas con clases originales
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)


# Asegurar carpeta de salida
out_dir = Path("figuras")
out_dir.mkdir(exist_ok=True)

# Tabla de conteo: filas = clase agrupada, columnas = clase original
tabla_stack = pd.crosstab(df["class_grouped"], df["type"])

# Orden deseado de clases agrupadas
orden_grupos = ["LPV", "BINARY", "PULS", "OTHER"]
tabla_stack = tabla_stack.reindex(orden_grupos)

# Quitar columnas vacías, por si acaso
tabla_stack = tabla_stack.loc[:, tabla_stack.sum(axis=0) > 0]

print("Tabla de clases originales dentro de cada clase agrupada:")
print(tabla_stack)

# Paleta personalizada
colors = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}

# Colores en el orden de las columnas reales
bar_colors = [colors.get(col, MUTED) for col in tabla_stack.columns]

# Crear figura
fig, ax = plt.subplots(figsize=(11, 6))

bottom = None

for i, col in enumerate(tabla_stack.columns):
    values = tabla_stack[col].values

    if bottom is None:
        bottom = [0] * len(values)

    bars = ax.bar(
        tabla_stack.index,
        values,
        bottom=bottom,
        label=f"{col} ({int(values.sum())})",
        color=bar_colors[i],
        edgecolor=LIGHT,
        linewidth=1
    )


    for bar, value, base in zip(bars, values, bottom):
        if value > 0:
            y_pos = base + value / 2

            # Para evitar texto en segmentos muy pequeños
            if value >= 20:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    str(int(value)),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color=LIGHT,
                    fontweight="bold"
                )

    bottom = [b + v for b, v in zip(bottom, values)]

# Total encima de cada barra agrupada
totales = tabla_stack.sum(axis=1)

for x, total in enumerate(totales):
    ax.text(
        x,
        total + 40,
        f"Total: {int(total)}",
        ha="center",
        va="bottom",
        fontsize=10,
        color=INK,
        fontweight="bold"
    )

# Estética
ax.set_title(
    "Distribución de clases agrupadas con composición interna - OGLE-IV GSEP",
    fontsize=14,
    color=INK
)

ax.set_xlabel("Clase agrupada", fontsize=12, color=INK)
ax.set_ylabel("Número de objetos", fontsize=12, color=INK)

ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.xticks(rotation=0)

# Leyenda con cantidad total de cada clase original
ax.legend(
    title="Clase original dentro del grupo",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

plt.tight_layout()

grouped_png = out_dir / "ogle_histograma_clases_agrupadas_apilado.png"
plt.savefig(grouped_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nFigura guardada en:")
print(grouped_png.resolve())


<!-- LABDOC_AUTO -->
### 4. Features base del cat?logo y de la curva I

- **Teor?a:** Las curvas de luz pueden resumirse con amplitud, dispersi?n, asimetr?a, curtosis, MAD, percentiles y relaciones de per?odo para alimentar modelos tabulares.
- **Llega:** `list.dat` y fotometr?a individual en `phot/I/*.dat`.
- **Sale:** `features_ogle_I.csv`, histogramas por clase y resumen apilado.
- **Insights:** Las variables ligadas a per?odo, amplitud y forma aportan separaci?n ?til; medias y medianas se calculan para an?lisis, pero se excluyen de la selecci?n final por restricci?n del experimento.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)

from pathlib import Path



# ============================================================
# 0) Ruta del directorio descomprimido
# ============================================================

data_dir = Path("OGLE4-GSEP-full")

if not data_dir.exists():
    raise FileNotFoundError(f"No encontre el directorio descomprimido en:\n{data_dir.resolve()}")

matches = list(data_dir.rglob("list.dat"))

if len(matches) == 0:
    raise FileNotFoundError(f"No encontre list.dat dentro de:\n{data_dir.resolve()}")

list_path = matches[0]
base_dir = list_path.parent
phot_I_dir = base_dir / "phot" / "I"

print("Directorio de datos:")
print(data_dir.resolve())

print("\nUsando carpeta base:")
print(base_dir)

print("\nRuta de list.dat:")
print(list_path)

print("\nRuta de curvas de luz I:")
print(phot_I_dir)

if not phot_I_dir.exists():
    raise FileNotFoundError(f"No encontre la carpeta phot/I en:\n{phot_I_dir}")

out_dir = Path("feature_histograms_ogle")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = Path("features_ogle_I.csv")

# ============================================================
# 1) Leer list.dat
# ============================================================

colspecs = [
    (0, 15),    # OGLE-IV ID
    (17, 28),   # RA J2000
    (29, 40),   # Dec J2000
    (41, 46),   # Variability type
    (47, 52),   # Subtype
    (54, 60),   # I mean mag
    (61, 67),   # V mean mag
    (68, 80),   # Period
    (81, 86),   # I amplitude
    (87, 99),   # Secondary period
]

cols = [
    "id", "ra", "dec", "type", "subtype",
    "I_mean", "V_mean", "period", "I_amp", "period2"
]

df = pd.read_fwf(
    list_path,
    colspecs=colspecs,
    names=cols,
    header=None
)

# Limpieza básica
for c in ["id", "ra", "dec", "type", "subtype"]:
    df[c] = df[c].astype(str).str.strip()
    df.loc[df[c].isin(["", "nan", "None"]), c] = np.nan

for c in ["I_mean", "V_mean", "period", "I_amp", "period2"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\nNúmero total de objetos:")
print(len(df))

print("\nClases originales:")
print(df["type"].value_counts())

# ============================================================
# 2) Agrupar clases similares
# ============================================================

def agrupar_clases_ogle(tipo):
    """
    Agrupa clases del atlas OGLE en categorías más generales.
    """

    if tipo in ["DCEP", "T2CEP", "ACEP", "RRLYR", "DSCT"]:
        return "PULS"

    elif tipo in ["ECL", "ELL"]:
        return "BINARY"

    elif tipo == "LPV":
        return "LPV"

    elif tipo == "OTHER":
        return "OTHER"

    else:
        return "UNKNOWN"


df["class_grouped"] = df["type"].apply(agrupar_clases_ogle)

print("\nDistribución de clases agrupadas:")
print(df["class_grouped"].value_counts())

# ============================================================
# 3) Función segura para kurtosis
# ============================================================

def safe_kurtosis(values):
    """
    Calcula kurtosis de Fisher usando pandas.
    Si hay pocos datos o todos los valores son iguales, devuelve NaN.
    En pandas, una distribución normal tiene kurtosis cercana a 0.
    """

    values = pd.Series(values).dropna()

    if len(values) < 4:
        return np.nan

    if values.nunique() <= 1:
        return np.nan

    return values.kurt()

# ============================================================
# 4) Calcular features desde curvas de luz en banda I
# ============================================================

features_from_lightcurve = []

for i, row in df.iterrows():

    object_id = row["id"]
    file_i = phot_I_dir / f"{object_id}.dat"

    if not file_i.exists():
        features_from_lightcurve.append({
            "id": object_id,
            "n_obs_I": np.nan,
            "I_median": np.nan,
            "I_p95_p5": np.nan,
            "I_mad": np.nan,
            "I_kurtosis": np.nan,
        })
        continue

    try:
        data = np.loadtxt(file_i)

        if data.ndim == 1:
            data = data.reshape(1, -1)

        # En los archivos phot/I:
        # columna 0 = tiempo
        # columna 1 = magnitud I
        # columna 2 = error fotométrico, si existe
        mag = data[:, 1]

        med = np.nanmedian(mag)

        features_from_lightcurve.append({
            "id": object_id,
            "n_obs_I": len(mag),
            "I_median": med,
            "I_p95_p5": np.nanpercentile(mag, 95) - np.nanpercentile(mag, 5),
            "I_mad": np.nanmedian(np.abs(mag - med)),
            "I_kurtosis": safe_kurtosis(mag),
        })

    except Exception:
        features_from_lightcurve.append({
            "id": object_id,
            "n_obs_I": np.nan,
            "I_median": np.nan,
            "I_p95_p5": np.nan,
            "I_mad": np.nan,
            "I_kurtosis": np.nan,
        })

    if (i + 1) % 500 == 0:
        print(f"Procesadas {i + 1} curvas de luz de {len(df)}")

df_lc = pd.DataFrame(features_from_lightcurve)

# Unir features del catálogo y de curvas de luz
df_feat = df.merge(df_lc, on="id", how="left")

# ============================================================
# 5) Crear features finales
# ============================================================

# Color V-I
# En magnitudes astronómicas:
# V_minus_I mayor generalmente significa objeto más rojo.
df_feat["V_minus_I"] = df_feat["V_mean"] - df_feat["I_mean"]

# Periodos en escala logarítmica
df_feat["log_period"] = np.log10(df_feat["period"].where(df_feat["period"] > 0))
df_feat["log_period2"] = np.log10(df_feat["period2"].where(df_feat["period2"] > 0))

# Relación de periodos
df_feat["period_ratio"] = df_feat["period2"] / df_feat["period"]

# Indicador de periodo secundario
df_feat["has_period2"] = df_feat["period2"].notna().astype(int)

# ============================================================
# 6) Lista final de 10 features
# ============================================================

feature_cols = [
    "I_mean",
    "V_minus_I",
    "log_period",
    "I_amp",
    "log_period2",
    "period_ratio",
    "has_period2",
    "I_p95_p5",
    "I_mad",
    "I_kurtosis",
]

# Guardar tabla
df_feat[["id", "type", "subtype", "class_grouped", "n_obs_I", "I_median"] + feature_cols].to_csv(
    csv_path,
    index=False
)

print("\nTabla de features guardada en:")
print(csv_path)

print("\nFeatures finales usadas para el modelo:")
for f in feature_cols:
    print("-", f)

print("\nPorcentaje de valores faltantes por feature:")
print((df_feat[feature_cols].isna().mean() * 100).round(2))

print("\nMedianas por clase agrupada:")
print(df_feat.groupby("class_grouped")[feature_cols].median().round(4))

# ============================================================
# 7) Histogramas para evaluar separación
# ============================================================

CLASS_COLORS = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
}

CLASS_ORDER = ["LPV", "BINARY", "PULS", "OTHER"]


def plot_continuous_feature(df_plot, feature, out_dir, bins=35):
    """
    Histograma normalizado por clase.
    Sirve para comparar separación sin que las clases grandes dominen.
    """

    plt.figure(figsize=(9, 5))

    for cls in CLASS_ORDER:
        values = df_plot.loc[df_plot["class_grouped"] == cls, feature].dropna()

        if len(values) == 0:
            continue

        plt.hist(
            values,
            bins=bins,
            density=True,
            histtype="step",
            linewidth=2,
            color=CLASS_COLORS[cls],
            label=f"{cls} (n={len(values)})"
        )

    plt.title(f"Separación por clase usando {feature}", fontsize=13)
    plt.xlabel(feature)
    plt.ylabel("Densidad normalizada")
    plt.grid(axis="y", alpha=0.25)
    plt.legend(title="Clase agrupada")
    plt.tight_layout()

    out_path = out_dir / f"hist_{feature}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    return out_path


def plot_binary_feature(df_plot, feature, out_dir):
    """
    Gráfico de proporción para variable binaria.
    """

    counts = pd.crosstab(
        df_plot["class_grouped"],
        df_plot[feature],
        normalize="index"
    ).reindex(CLASS_ORDER)

    plt.figure(figsize=(8, 5))

    x = np.arange(len(counts.index))
    width = 0.35

    values_0 = counts.get(0, pd.Series(0, index=counts.index)).values
    values_1 = counts.get(1, pd.Series(0, index=counts.index)).values

    plt.bar(
        x - width / 2,
        values_0,
        width,
        label=f"{feature}=0",
        color=LIGHT,
        edgecolor=PRIMARY,
        linewidth=1
    )

    plt.bar(
        x + width / 2,
        values_1,
        width,
        label=f"{feature}=1",
        color=PRIMARY,
        edgecolor=DARK,
        linewidth=1
    )

    plt.xticks(x, counts.index)
    plt.ylabel("Proporción dentro de cada clase")
    plt.xlabel("Clase agrupada")
    plt.title(f"Proporción de {feature} por clase", fontsize=13)
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.25)
    plt.legend()
    plt.tight_layout()

    out_path = out_dir / f"hist_{feature}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    return out_path


saved_paths = []

for feature in feature_cols:

    if feature == "has_period2":
        path = plot_binary_feature(df_feat, feature, out_dir)
    else:
        path = plot_continuous_feature(df_feat, feature, out_dir)

    saved_paths.append(path)

print("\nHistogramas guardados en:")
for p in saved_paths:
    print(p)

# ============================================================
# 8) Histograma apilado de clases agrupadas
# ============================================================

tabla_stack = pd.crosstab(df_feat["class_grouped"], df_feat["type"])

orden_grupos = ["LPV", "BINARY", "PULS", "OTHER"]
tabla_stack = tabla_stack.reindex(orden_grupos)

tabla_stack = tabla_stack.loc[:, tabla_stack.sum(axis=0) > 0]

print("\nTabla de clases originales dentro de cada clase agrupada:")
print(tabla_stack)

colors_original = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}

bar_colors = [colors_original.get(col, MUTED) for col in tabla_stack.columns]

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(tabla_stack.index))

for i, col in enumerate(tabla_stack.columns):
    values = tabla_stack[col].values

    bars = ax.bar(
        tabla_stack.index,
        values,
        bottom=bottom,
        label=f"{col} ({int(values.sum())})",
        color=bar_colors[i],
        edgecolor=LIGHT,
        linewidth=1
    )

    for bar, value, base in zip(bars, values, bottom):
        if value >= 20:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                str(int(value)),
                ha="center",
                va="center",
                fontsize=9,
                color=LIGHT,
                fontweight="bold"
            )

    bottom += values

totales = tabla_stack.sum(axis=1)

for x, total in enumerate(totales):
    ax.text(
        x,
        total + 40,
        f"Total: {int(total)}",
        ha="center",
        va="bottom",
        fontsize=10,
        color=INK,
        fontweight="bold"
    )

ax.set_title(
    "Distribución de clases agrupadas con composición interna - OGLE-IV GSEP",
    fontsize=14,
    color=INK
)

ax.set_xlabel("Clase agrupada")
ax.set_ylabel("Número de objetos")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    title="Clase original",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

plt.tight_layout()

stacked_path = out_dir / "hist_clases_agrupadas_apilado.png"
plt.savefig(stacked_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nHistograma apilado guardado en:")
print(stacked_path)


<!-- LABDOC_AUTO -->
### 5. Per?odo Lomb-Scargle y color I/V

- **Teor?a:** Lomb-Scargle estima per?odos en series temporales irregulares; el color `V-I` resume temperatura/apariencia fotom?trica y ayuda a separar familias f?sicas.
- **Llega:** `features_ogle_I.csv`, curvas `phot/I` y `phot/V`, m?s el per?odo catalogado.
- **Sale:** `photometry_lombscargle_color_features.csv` y `features_ogle_I_V_lombscargle.csv`.
- **Insights:** `ls_period_catalog_ratio` captura consistencia entre per?odo observado y catalogado; `V_minus_I` queda como feature permitida porque es color, no media/mediana individual.


In [ ]:
# ============================================================
# Periodo Lomb-Scargle y color fotometrico I/V
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import lombscargle

try:
    from IPython.display import display
except ImportError:
    display = print

# Si tienes OGLE3-GSEP-full en vez de OGLE4-GSEP-full, el codigo lo usa primero.
DATA_DIR_CANDIDATES = [
    Path("OGLE3-GSEP-full"),
    Path("OGLE4-GSEP-full"),
]

data_dir = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if data_dir is None:
    raise FileNotFoundError(
        "No encontre OGLE3-GSEP-full ni OGLE4-GSEP-full en el proyecto."
    )

phot_I_dir = data_dir / "phot" / "I"
phot_V_dir = data_dir / "phot" / "V"

if not phot_I_dir.exists():
    raise FileNotFoundError(f"No encontre el directorio de fotometria I en:\n{phot_I_dir.resolve()}")

if not phot_V_dir.exists():
    raise FileNotFoundError(f"No encontre el directorio de fotometria V en:\n{phot_V_dir.resolve()}")

base_csv_path = Path("features_ogle_I.csv")
extended_csv_path = Path("features_ogle_I_V_lombscargle.csv")
phot_features_csv_path = Path("photometry_lombscargle_color_features.csv")

# Cambia a True si quieres recalcular aunque el CSV extendido ya exista.
RECOMPUTE_LOMB_SCARGLE = False

# Parametros del periodograma. Aumentar LS_N_FREQUENCIES mejora precision pero tarda mas.
LS_MIN_PERIOD_DAYS = 0.05
LS_MAX_PERIOD_DAYS = 1000.0
LS_N_FREQUENCIES = 12000
LS_N_REFINE = 1200
LS_REFINE_WIDTH_STEPS = 5
LS_MIN_OBS = 20
MAX_OBJECTS = None  # Usa un entero pequeno para pruebas rapidas, por ejemplo 100.

if extended_csv_path.exists() and not RECOMPUTE_LOMB_SCARGLE:
    df_feat_extended = pd.read_csv(extended_csv_path)
    print("Dataframe extendido cargado desde:")
    print(extended_csv_path)
else:
    if "df_feat" in globals():
        df_base = df_feat.copy()
    elif base_csv_path.exists():
        df_base = pd.read_csv(base_csv_path)
    else:
        raise FileNotFoundError(
            "No encontre df_feat en memoria ni features_ogle_I.csv. "
            "Ejecuta primero la celda que construye el dataframe de features base."
        )

    df_base["id"] = df_base["id"].astype(str).str.strip()

    def index_photometry_files(root_dir, forbidden_band=None):
        """
        Indexa archivos .dat por ID. Si hay duplicados por carpetas anidadas,
        conserva la ruta mas corta, que corresponde al archivo canonico.
        """

        files_by_id = {}

        for path in root_dir.rglob("*.dat"):
            rel_parts = path.relative_to(root_dir).parts[:-1]

            if forbidden_band is not None and forbidden_band in rel_parts:
                continue

            object_id = path.stem
            previous = files_by_id.get(object_id)

            if previous is None or len(path.parts) < len(previous.parts):
                files_by_id[object_id] = path

        return files_by_id

    def read_photometry(path):
        if path is None or not path.exists():
            return np.array([]), np.array([]), np.array([])

        try:
            data = np.loadtxt(path)
        except Exception:
            return np.array([]), np.array([]), np.array([])

        if data.ndim == 1:
            data = data.reshape(1, -1)

        if data.shape[1] < 2:
            return np.array([]), np.array([]), np.array([])

        time = data[:, 0].astype(float)
        mag = data[:, 1].astype(float)

        if data.shape[1] >= 3:
            err = data[:, 2].astype(float)
        else:
            err = np.full_like(mag, np.nan, dtype=float)

        valid = np.isfinite(time) & np.isfinite(mag)
        return time[valid], mag[valid], err[valid]

    def robust_weighted_mean(mag, err):
        valid = np.isfinite(mag) & np.isfinite(err) & (err > 0)

        if valid.sum() == 0:
            return np.nan

        weights = 1.0 / np.square(err[valid])
        return np.average(mag[valid], weights=weights)

    def photometry_summary(time, mag, err):
        if len(mag) == 0:
            return {
                "n_obs": 0,
                "mean": np.nan,
                "median": np.nan,
                "weighted_mean": np.nan,
                "std": np.nan,
                "mad": np.nan,
                "p95_p5": np.nan,
            }

        median = np.nanmedian(mag)

        return {
            "n_obs": len(mag),
            "mean": np.nanmean(mag),
            "median": median,
            "weighted_mean": robust_weighted_mean(mag, err),
            "std": np.nanstd(mag, ddof=1) if len(mag) > 1 else np.nan,
            "mad": np.nanmedian(np.abs(mag - median)),
            "p95_p5": np.nanpercentile(mag, 95) - np.nanpercentile(mag, 5),
        }

    def estimate_lomb_scargle_period(time, mag):
        if len(time) < LS_MIN_OBS:
            return {
                "ls_period_I": np.nan,
                "ls_log_period_I": np.nan,
                "ls_frequency_I": np.nan,
                "ls_power_I": np.nan,
            }

        order = np.argsort(time)
        time = np.asarray(time[order], dtype=float)
        mag = np.asarray(mag[order], dtype=float)

        time = time - np.nanmin(time)
        mag_centered = mag - np.nanmedian(mag)
        mag_std = np.nanstd(mag_centered)

        if not np.isfinite(mag_std) or mag_std == 0:
            return {
                "ls_period_I": np.nan,
                "ls_log_period_I": np.nan,
                "ls_frequency_I": np.nan,
                "ls_power_I": np.nan,
            }

        mag_centered = mag_centered / mag_std
        baseline = np.nanmax(time) - np.nanmin(time)
        max_period = min(LS_MAX_PERIOD_DAYS, 0.9 * baseline)

        if not np.isfinite(max_period) or max_period <= LS_MIN_PERIOD_DAYS:
            return {
                "ls_period_I": np.nan,
                "ls_log_period_I": np.nan,
                "ls_frequency_I": np.nan,
                "ls_power_I": np.nan,
            }

        min_frequency = 1.0 / max_period
        max_frequency = 1.0 / LS_MIN_PERIOD_DAYS
        frequencies = np.linspace(min_frequency, max_frequency, LS_N_FREQUENCIES)
        angular_frequencies = 2.0 * np.pi * frequencies

        power = lombscargle(
            time,
            mag_centered,
            angular_frequencies,
            precenter=False,
            normalize=True,
        )

        if not np.isfinite(power).any():
            return {
                "ls_period_I": np.nan,
                "ls_log_period_I": np.nan,
                "ls_frequency_I": np.nan,
                "ls_power_I": np.nan,
            }

        best_idx = int(np.nanargmax(power))
        best_frequency = frequencies[best_idx]
        best_power = power[best_idx]

        if LS_N_REFINE > 0 and 0 < best_idx < len(frequencies) - 1:
            frequency_step = frequencies[1] - frequencies[0]
            fine_min = max(min_frequency, best_frequency - LS_REFINE_WIDTH_STEPS * frequency_step)
            fine_max = min(max_frequency, best_frequency + LS_REFINE_WIDTH_STEPS * frequency_step)
            fine_frequencies = np.linspace(fine_min, fine_max, LS_N_REFINE)
            fine_power = lombscargle(
                time,
                mag_centered,
                2.0 * np.pi * fine_frequencies,
                precenter=False,
                normalize=True,
            )
            fine_best_idx = int(np.nanargmax(fine_power))
            best_frequency = fine_frequencies[fine_best_idx]
            best_power = fine_power[fine_best_idx]

        best_period = 1.0 / best_frequency

        return {
            "ls_period_I": best_period,
            "ls_log_period_I": np.log10(best_period) if best_period > 0 else np.nan,
            "ls_frequency_I": best_frequency,
            "ls_power_I": best_power,
        }

    i_files = index_photometry_files(phot_I_dir, forbidden_band="V")
    v_files = index_photometry_files(phot_V_dir, forbidden_band="I")

    print("Directorio de datos:", data_dir.resolve())
    print("Archivos I indexados:", len(i_files))
    print("Archivos V indexados:", len(v_files))

    object_ids = df_base["id"].dropna().astype(str).tolist()

    if MAX_OBJECTS is not None:
        object_ids = object_ids[:MAX_OBJECTS]

    rows = []

    for index, object_id in enumerate(object_ids, start=1):
        i_path = i_files.get(object_id)
        v_path = v_files.get(object_id)

        time_i, mag_i, err_i = read_photometry(i_path)
        time_v, mag_v, err_v = read_photometry(v_path)

        stats_i = photometry_summary(time_i, mag_i, err_i)
        stats_v = photometry_summary(time_v, mag_v, err_v)
        ls_features = estimate_lomb_scargle_period(time_i, mag_i)

        row = {
            "id": object_id,
            "phot_I_path": str(i_path) if i_path is not None else np.nan,
            "phot_V_path": str(v_path) if v_path is not None else np.nan,
            "n_obs_I_phot": stats_i["n_obs"],
            "I_phot_mean": stats_i["mean"],
            "I_phot_median": stats_i["median"],
            "I_phot_weighted_mean": stats_i["weighted_mean"],
            "I_phot_std": stats_i["std"],
            "I_phot_mad": stats_i["mad"],
            "I_phot_p95_p5": stats_i["p95_p5"],
            "n_obs_V": stats_v["n_obs"],
            "V_phot_mean": stats_v["mean"],
            "V_phot_median": stats_v["median"],
            "V_phot_weighted_mean": stats_v["weighted_mean"],
            "V_phot_std": stats_v["std"],
            "V_phot_mad": stats_v["mad"],
            "V_phot_p95_p5": stats_v["p95_p5"],
            "V_minus_I_phot_mean": stats_v["mean"] - stats_i["mean"],
            "V_minus_I_phot_median": stats_v["median"] - stats_i["median"],
            "V_minus_I_phot_weighted_mean": stats_v["weighted_mean"] - stats_i["weighted_mean"],
        }

        row.update(ls_features)
        rows.append(row)

        if index % 500 == 0 or index == len(object_ids):
            print(f"Procesadas {index} de {len(object_ids)} estrellas")

    df_phot_ls = pd.DataFrame(rows)

    if "log_period" in df_base.columns:
        df_phot_ls = df_phot_ls.merge(
            df_base[["id", "log_period"]].copy(),
            on="id",
            how="left",
        )
        catalog_period = np.power(10.0, df_phot_ls["log_period"])
        df_phot_ls["ls_period_catalog_ratio"] = df_phot_ls["ls_period_I"] / catalog_period
        df_phot_ls["ls_log_period_delta_catalog"] = df_phot_ls["ls_log_period_I"] - df_phot_ls["log_period"]
        df_phot_ls = df_phot_ls.drop(columns=["log_period"])

    new_feature_columns = [col for col in df_phot_ls.columns if col != "id"]
    df_base_without_old = df_base.drop(
        columns=[col for col in new_feature_columns if col in df_base.columns],
        errors="ignore",
    )

    df_feat_extended = df_base_without_old.merge(df_phot_ls, on="id", how="left")

    df_phot_ls.to_csv(phot_features_csv_path, index=False)
    df_feat_extended.to_csv(extended_csv_path, index=False)

    print("\nFeatures fotometricas guardadas en:")
    print(phot_features_csv_path)
    print("\nDataframe final extendido guardado en:")
    print(extended_csv_path)

print("\nColumnas nuevas disponibles:")
new_columns_to_show = [
    "ls_period_I",
    "ls_log_period_I",
    "ls_power_I",
    "ls_period_catalog_ratio",
    "ls_log_period_delta_catalog",
    "V_minus_I_phot_mean",
    "V_minus_I_phot_median",
    "V_minus_I_phot_weighted_mean",
    "n_obs_V",
    "I_phot_std",
    "V_phot_std",
]
print([col for col in new_columns_to_show if col in df_feat_extended.columns])

print("\nVista rapida:")
preview_cols = ["id", "type", "class_grouped"] + [
    col for col in new_columns_to_show if col in df_feat_extended.columns
]
display(df_feat_extended[preview_cols].head())


<!-- LABDOC_AUTO -->
### 6. Features de Fourier sobre la banda I

- **Teor?a:** Una serie de Fourier sobre la curva plegada describe forma peri?dica mediante amplitudes, fases y razones arm?nicas independientes del nivel absoluto.
- **Llega:** Tabla con Lomb-Scargle/color y archivos `phot/I`.
- **Sale:** `fourier_I_features.csv` y `features_ogle_I_V_lombscargle_fourier.csv`.
- **Insights:** Fourier es informativo cuando la curva est? bien plegada y tiene forma peri?dica estable; por eso despu?s se a?ade una feature expl?cita de calidad del fit.


In [ ]:
# ============================================================
# Features de Fourier para describir la curva de luz en banda I
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR_CANDIDATES = [
    Path("OGLE3-GSEP-full"),
    Path("OGLE4-GSEP-full"),
]

data_dir = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if data_dir is None:
    raise FileNotFoundError(
        "No encontre OGLE3-GSEP-full ni OGLE4-GSEP-full en el proyecto."
    )

phot_I_dir = data_dir / "phot" / "I"

if not phot_I_dir.exists():
    raise FileNotFoundError(f"No encontre el directorio de fotometria I en:\n{phot_I_dir.resolve()}")

base_csv_path = Path("features_ogle_I.csv")
lomb_color_csv_path = Path("features_ogle_I_V_lombscargle.csv")
fourier_features_csv_path = Path("fourier_I_features.csv")
final_fourier_csv_path = Path("features_ogle_I_V_lombscargle_fourier.csv")

# Cambia a True si quieres recalcular aunque el CSV final ya exista.
RECOMPUTE_FOURIER = False

# Cuatro armonicos suelen capturar forma general sin sobreajustar demasiado.
FOURIER_N_HARMONICS = 4
FOURIER_MIN_OBS = 2 * FOURIER_N_HARMONICS + 5
MAX_OBJECTS_FOURIER = None  # Usa un entero pequeno, por ejemplo 100, para pruebas rapidas.

if final_fourier_csv_path.exists() and not RECOMPUTE_FOURIER:
    df_feat_fourier = pd.read_csv(final_fourier_csv_path)
    print("Dataframe con features Fourier cargado desde:")
    print(final_fourier_csv_path)
else:
    if "df_feat_extended" in globals():
        df_base_fourier = df_feat_extended.copy()
    elif lomb_color_csv_path.exists():
        df_base_fourier = pd.read_csv(lomb_color_csv_path)
    elif "df_feat" in globals():
        df_base_fourier = df_feat.copy()
    elif base_csv_path.exists():
        df_base_fourier = pd.read_csv(base_csv_path)
    else:
        raise FileNotFoundError(
            "No encontre un dataframe base. Ejecuta primero las celdas de features."
        )

    df_base_fourier["id"] = df_base_fourier["id"].astype(str).str.strip()

    def index_i_photometry_files(root_dir):
        """
        Indexa archivos I por ID. Si hay duplicados por extracciones anidadas,
        conserva la ruta mas corta y descarta carpetas V colgadas dentro de I.
        """

        files_by_id = {}

        for path in root_dir.rglob("*.dat"):
            rel_parts = path.relative_to(root_dir).parts[:-1]

            if "V" in rel_parts:
                continue

            object_id = path.stem
            previous = files_by_id.get(object_id)

            if previous is None or len(path.parts) < len(previous.parts):
                files_by_id[object_id] = path

        return files_by_id

    def read_i_photometry(path):
        if path is None or not path.exists():
            return np.array([]), np.array([]), np.array([])

        try:
            data = np.loadtxt(path)
        except Exception:
            return np.array([]), np.array([]), np.array([])

        if data.ndim == 1:
            data = data.reshape(1, -1)

        if data.shape[1] < 2:
            return np.array([]), np.array([]), np.array([])

        time = data[:, 0].astype(float)
        mag = data[:, 1].astype(float)

        if data.shape[1] >= 3:
            err = data[:, 2].astype(float)
        else:
            err = np.full_like(mag, np.nan, dtype=float)

        valid = np.isfinite(time) & np.isfinite(mag)
        return time[valid], mag[valid], err[valid]

    def choose_fourier_period(row):
        """
        Prioriza el periodo Lomb-Scargle si ya fue calculado; si no existe,
        usa el periodo del catalogo a partir de log_period.
        """

        ls_period = row.get("ls_period_I", np.nan)

        if np.isfinite(ls_period) and ls_period > 0:
            return float(ls_period), "lomb_scargle"

        log_period = row.get("log_period", np.nan)

        if np.isfinite(log_period):
            period = float(np.power(10.0, log_period))

            if np.isfinite(period) and period > 0:
                return period, "catalog"

        return np.nan, "missing"

    def empty_fourier_features(n_harmonics=FOURIER_N_HARMONICS):
        features = {
            "fourier_period_I": np.nan,
            "fourier_period_source_I": "missing",
            "fourier_phase_coverage_I": np.nan,
            "fourier_offset_I": np.nan,
            "fourier_model_amp_I": np.nan,
            "fourier_total_amp_I": np.nan,
            "fourier_resid_std_I": np.nan,
            "fourier_chi2_red_I": np.nan,
            "fourier_var_ratio_I": np.nan,
        }

        for harmonic in range(1, n_harmonics + 1):
            features[f"fourier_A{harmonic}_I"] = np.nan
            features[f"fourier_phi{harmonic}_I"] = np.nan

        for harmonic in range(2, n_harmonics + 1):
            features[f"fourier_R{harmonic}1_I"] = np.nan
            features[f"fourier_phi{harmonic}1_I"] = np.nan

        return features

    def fit_fourier_features(time, mag, err, period, period_source):
        features = empty_fourier_features()
        features["fourier_period_I"] = period
        features["fourier_period_source_I"] = period_source

        if len(time) < FOURIER_MIN_OBS or not np.isfinite(period) or period <= 0:
            return features

        order = np.argsort(time)
        time = np.asarray(time[order], dtype=float)
        mag = np.asarray(mag[order], dtype=float)
        err = np.asarray(err[order], dtype=float) if len(err) == len(mag) else np.full_like(mag, np.nan)

        phase = ((time - np.nanmin(time)) / period) % 1.0
        y = mag - np.nanmedian(mag)

        valid = np.isfinite(phase) & np.isfinite(y)
        phase = phase[valid]
        y = y[valid]
        err = err[valid]

        if len(y) < FOURIER_MIN_OBS or np.nanstd(y) == 0:
            return features

        phase_bins = pd.cut(phase, bins=np.linspace(0, 1, 21), include_lowest=True)
        features["fourier_phase_coverage_I"] = phase_bins.value_counts().gt(0).mean()

        design_columns = [np.ones_like(phase)]

        for harmonic in range(1, FOURIER_N_HARMONICS + 1):
            angle = 2.0 * np.pi * harmonic * phase
            design_columns.append(np.cos(angle))
            design_columns.append(np.sin(angle))

        design = np.column_stack(design_columns)
        valid_err = np.isfinite(err) & (err > 0)

        if valid_err.sum() >= 0.5 * len(y):
            err_floor = np.nanmedian(err[valid_err]) * 0.2
            err_ceiling = np.nanpercentile(err[valid_err], 95) * 5.0
            err_clipped = np.clip(err, err_floor, err_ceiling)
            weights = 1.0 / err_clipped
            lhs = design * weights[:, None]
            rhs = y * weights
        else:
            lhs = design
            rhs = y

        try:
            coefficients, *_ = np.linalg.lstsq(lhs, rhs, rcond=None)
        except np.linalg.LinAlgError:
            return features

        model = design @ coefficients
        residuals = y - model

        amplitudes = []
        phases = []

        for harmonic in range(1, FOURIER_N_HARMONICS + 1):
            cos_coef = coefficients[2 * harmonic - 1]
            sin_coef = coefficients[2 * harmonic]
            amplitude = np.hypot(cos_coef, sin_coef)
            phase_angle = np.arctan2(-sin_coef, cos_coef) % (2.0 * np.pi)

            amplitudes.append(amplitude)
            phases.append(phase_angle)

            features[f"fourier_A{harmonic}_I"] = amplitude
            features[f"fourier_phi{harmonic}_I"] = phase_angle

        a1 = amplitudes[0]

        if np.isfinite(a1) and a1 > 0:
            for harmonic in range(2, FOURIER_N_HARMONICS + 1):
                features[f"fourier_R{harmonic}1_I"] = amplitudes[harmonic - 1] / a1
                features[f"fourier_phi{harmonic}1_I"] = (
                    phases[harmonic - 1] - harmonic * phases[0]
                ) % (2.0 * np.pi)

        features["fourier_offset_I"] = np.nanmedian(mag) + coefficients[0]
        features["fourier_model_amp_I"] = np.nanmax(model) - np.nanmin(model)
        features["fourier_total_amp_I"] = np.sqrt(np.nansum(np.square(amplitudes)))
        features["fourier_resid_std_I"] = np.nanstd(residuals, ddof=1)

        degrees_of_freedom = max(len(y) - len(coefficients), 1)

        if valid_err.sum() >= 0.5 * len(y):
            features["fourier_chi2_red_I"] = np.nansum(np.square(residuals / err_clipped)) / degrees_of_freedom

        variance_y = np.nanvar(y)
        variance_residuals = np.nanvar(residuals)

        if np.isfinite(variance_y) and variance_y > 0:
            features["fourier_var_ratio_I"] = 1.0 - variance_residuals / variance_y

        return features

    i_files = index_i_photometry_files(phot_I_dir)

    print("Directorio de datos:", data_dir.resolve())
    print("Archivos I indexados:", len(i_files))

    object_rows = df_base_fourier.copy()

    if MAX_OBJECTS_FOURIER is not None:
        object_rows = object_rows.head(MAX_OBJECTS_FOURIER)

    rows = []

    for index, row in enumerate(object_rows.itertuples(index=False), start=1):
        row_dict = row._asdict()
        object_id = str(row_dict["id"]).strip()
        period, period_source = choose_fourier_period(row_dict)
        i_path = i_files.get(object_id)
        time_i, mag_i, err_i = read_i_photometry(i_path)

        fourier_features = fit_fourier_features(
            time=time_i,
            mag=mag_i,
            err=err_i,
            period=period,
            period_source=period_source,
        )

        fourier_features["id"] = object_id
        rows.append(fourier_features)

        if index % 500 == 0 or index == len(object_rows):
            print(f"Procesadas {index} de {len(object_rows)} estrellas")

    df_fourier_I = pd.DataFrame(rows)

    fourier_feature_columns = [col for col in df_fourier_I.columns if col != "id"]
    df_base_clean = df_base_fourier.drop(
        columns=[col for col in fourier_feature_columns if col in df_base_fourier.columns],
        errors="ignore",
    )

    df_feat_fourier = df_base_clean.merge(df_fourier_I, on="id", how="left")

    df_fourier_I.to_csv(fourier_features_csv_path, index=False)
    df_feat_fourier.to_csv(final_fourier_csv_path, index=False)

    print("\nFeatures Fourier guardadas en:")
    print(fourier_features_csv_path)
    print("\nDataframe final con Fourier guardado en:")
    print(final_fourier_csv_path)

print("\nFeatures Fourier principales:")
fourier_preview_columns = [
    "fourier_period_I",
    "fourier_period_source_I",
    "fourier_A1_I",
    "fourier_A2_I",
    "fourier_R21_I",
    "fourier_R31_I",
    "fourier_phi21_I",
    "fourier_phi31_I",
    "fourier_model_amp_I",
    "fourier_var_ratio_I",
    "fourier_phase_coverage_I",
]
print([col for col in fourier_preview_columns if col in df_feat_fourier.columns])

preview_cols = ["id", "type", "class_grouped"] + [
    col for col in fourier_preview_columns if col in df_feat_fourier.columns
]
display(df_feat_fourier[preview_cols].head())


<!-- LABDOC_AUTO -->
### 7. Calidad Fourier y GBM con top-k features

- **Teor?a:** Un GBM captura interacciones no lineales; el split 15/15/70 separa entrenamiento, selecci?n por validaci?n y prueba fuerte.
- **Llega:** Features completas con Fourier; se excluye `OTHER` y se filtran columnas con `mean`/`median` antes de rankear.
- **Sale:** `features_ogle_I_V_lombscargle_fourier_quality.csv`, rankings, m?tricas top 7-10 y matriz de confusi?n en `gbm_no_other_results/`.
- **Insights:** El top 8 fue el mejor por validaci?n; la calidad Fourier ayuda a que el modelo aprenda cu?ndo confiar en descriptores arm?nicos.


In [ ]:
# ============================================================
# GBM sin OTHER: calidad Fourier, split 15/15/70 y top 7-10
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)


from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
TARGET_COL = "class_grouped"
EXCLUDED_CLASS = "OTHER"
TRAIN_SIZE = 0.15
VALID_SIZE = 0.15
TEST_SIZE = 0.70
TOP_K_VALUES = [7, 8, 9, 10]
PERMUTATION_REPEATS_GBM = 8
MISSING_LIMIT = 85.0

source_candidates = [
    Path("features_ogle_I_V_lombscargle_fourier.csv"),
    Path("features_ogle_I_V_lombscargle.csv"),
    Path("features_ogle_I.csv"),
]

source_path = next((path for path in source_candidates if path.exists()), None)

if source_path is None:
    raise FileNotFoundError("No encontre ningun CSV de features para entrenar el GBM.")

df_gbm = pd.read_csv(source_path)

# ------------------------------------------------------------
# 1) Indicador de calidad del fit Fourier
# ------------------------------------------------------------

required_fourier_quality_inputs = [
    "fourier_var_ratio_I",
    "fourier_phase_coverage_I",
    "fourier_model_amp_I",
    "fourier_resid_std_I",
]
missing_quality_inputs = [
    col for col in required_fourier_quality_inputs
    if col not in df_gbm.columns
]

if missing_quality_inputs:
    raise ValueError(
        "No puedo calcular fourier_fit_quality_I porque faltan columnas: "
        f"{missing_quality_inputs}. Ejecuta primero la celda de features Fourier."
    )

var_score = df_gbm["fourier_var_ratio_I"].clip(lower=0, upper=1)
coverage_score = df_gbm["fourier_phase_coverage_I"].clip(lower=0, upper=1)
model_amp = df_gbm["fourier_model_amp_I"].clip(lower=0)
resid_std = df_gbm["fourier_resid_std_I"].clip(lower=0)
resid_score = model_amp / (model_amp + 2.0 * resid_std)
resid_score = resid_score.replace([np.inf, -np.inf], np.nan).fillna(0)

df_gbm["fourier_fit_quality_I"] = (
    var_score.fillna(0)
    * coverage_score.fillna(0)
    * resid_score.fillna(0)
).clip(lower=0, upper=1)

df_gbm["fourier_fit_reliable_I"] = (
    (var_score >= 0.50)
    & (coverage_score >= 0.80)
    & (resid_score >= 0.50)
).astype(int)

quality_csv_path = Path("features_ogle_I_V_lombscargle_fourier_quality.csv")
df_gbm.to_csv(quality_csv_path, index=False)

print("CSV con indicador de calidad Fourier guardado en:")
print(quality_csv_path)

print("\nResumen del indicador por clase:")
display(
    df_gbm.groupby(TARGET_COL)[[
        "fourier_fit_quality_I",
        "fourier_fit_reliable_I",
        "fourier_var_ratio_I",
        "fourier_phase_coverage_I",
    ]]
    .agg(["median", "mean", "count"])
    .round(4)
)

# ------------------------------------------------------------
# 2) Dataset supervisado: excluir OTHER y descartar medias/medianas
# ------------------------------------------------------------

if TARGET_COL not in df_gbm.columns:
    raise ValueError(f"Falta la columna target: {TARGET_COL}")

df_model = df_gbm.loc[df_gbm[TARGET_COL] != EXCLUDED_CLASS].copy()
df_model[TARGET_COL] = df_model[TARGET_COL].astype(str)

excluded_columns = {
    "id",
    "type",
    "subtype",
    TARGET_COL,
    "phot_I_path",
    "phot_V_path",
    "fourier_period_source_I",
}

# Por requisito del modelo final, ninguna feature seleccionable puede ser media o mediana.
# Esto excluye columnas con mean/median en el nombre, incluyendo weighted_mean.
# Conservamos V_minus_I porque es una feature de color, no una magnitud media individual.
def is_forbidden_mean_or_median_feature(feature):
    lower_name = feature.lower()
    return "mean" in lower_name or "median" in lower_name

numeric_features = [
    col for col in df_model.select_dtypes(include=[np.number]).columns
    if col not in excluded_columns
]

feature_precheck_gbm = pd.DataFrame({
    "feature": numeric_features,
    "missing_pct": df_model[numeric_features].isna().mean().values * 100,
    "n_unique": df_model[numeric_features].nunique(dropna=True).values,
})
feature_precheck_gbm["forbidden_mean_or_median"] = feature_precheck_gbm["feature"].apply(
    is_forbidden_mean_or_median_feature
)

candidate_features = feature_precheck_gbm.loc[
    (feature_precheck_gbm["missing_pct"] < MISSING_LIMIT)
    & (feature_precheck_gbm["n_unique"] > 1)
    & (~feature_precheck_gbm["forbidden_mean_or_median"]),
    "feature",
].tolist()

if "fourier_fit_quality_I" not in candidate_features:
    candidate_features.append("fourier_fit_quality_I")
if "fourier_fit_reliable_I" not in candidate_features:
    candidate_features.append("fourier_fit_reliable_I")

forbidden_candidates = feature_precheck_gbm.loc[
    feature_precheck_gbm["forbidden_mean_or_median"],
    "feature",
].tolist()

X = df_model[candidate_features].replace([np.inf, -np.inf], np.nan)
y = df_model[TARGET_COL]
ids = df_model["id"].astype(str) if "id" in df_model.columns else pd.Series(df_model.index.astype(str), index=df_model.index)
types = df_model["type"].astype(str) if "type" in df_model.columns else pd.Series("", index=df_model.index)

print("\nClases usadas para GBM, excluyendo OTHER:")
print(y.value_counts())
print("Features candidatas para GBM:", len(candidate_features))
print("Features descartadas por ser medias/medianas:")
print(forbidden_candidates)

# ------------------------------------------------------------
# 3) Split estratificado 15% train, 15% validacion, 70% test
# ------------------------------------------------------------

train_valid_size = TRAIN_SIZE + VALID_SIZE
valid_fraction_inside_train_valid = VALID_SIZE / train_valid_size

X_train_valid, X_test, y_train_valid, y_test, ids_train_valid, ids_test, types_train_valid, types_test = train_test_split(
    X,
    y,
    ids,
    types,
    train_size=train_valid_size,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid, ids_train, ids_valid, types_train, types_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    ids_train_valid,
    types_train_valid,
    test_size=valid_fraction_inside_train_valid,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "n_objects": [len(y_train), len(y_valid), len(y_test)],
    "fraction": [len(y_train) / len(y), len(y_valid) / len(y), len(y_test) / len(y)],
})

print("\nSplit usado:")
display(split_summary.round(4))
print("\nDistribucion por split:")
display(pd.DataFrame({
    "train": y_train.value_counts().sort_index(),
    "valid": y_valid.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}).fillna(0).astype(int))

# ------------------------------------------------------------
# 4) Ranking con train y permutation importance en validacion
# ------------------------------------------------------------

X_train_imputed = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X_train),
    columns=candidate_features,
    index=X_train.index,
)

discrete_features = [
    feature.startswith("has_")
    or feature.startswith("n_obs")
    or feature.endswith("_reliable_I")
    for feature in candidate_features
]

mutual_info_scores = mutual_info_classif(
    X_train_imputed,
    y_train,
    discrete_features=discrete_features,
    random_state=RANDOM_STATE,
)

mi_ranking = pd.DataFrame({
    "feature": candidate_features,
    "mutual_info_train": mutual_info_scores,
}).sort_values("mutual_info_train", ascending=False).reset_index(drop=True)
mi_ranking["rank_mutual_info"] = np.arange(1, len(mi_ranking) + 1)


def make_gbm_model():
    return HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        early_stopping=True,
        random_state=RANDOM_STATE,
    )

rank_model = make_gbm_model()
rank_model.fit(X_train, y_train)

permutation_scores = permutation_importance(
    rank_model,
    X_valid,
    y_valid,
    scoring="f1_macro",
    n_repeats=PERMUTATION_REPEATS_GBM,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_ranking = pd.DataFrame({
    "feature": candidate_features,
    "gbm_valid_permutation_f1_drop": permutation_scores.importances_mean,
    "gbm_valid_permutation_std": permutation_scores.importances_std,
}).sort_values("gbm_valid_permutation_f1_drop", ascending=False).reset_index(drop=True)
permutation_ranking["rank_gbm_valid_permutation"] = np.arange(1, len(permutation_ranking) + 1)

gbm_feature_ranking = mi_ranking.merge(permutation_ranking, on="feature")
gbm_feature_ranking["avg_rank"] = gbm_feature_ranking[[
    "rank_mutual_info",
    "rank_gbm_valid_permutation",
]].mean(axis=1)

gbm_feature_ranking = gbm_feature_ranking.sort_values(
    ["rank_gbm_valid_permutation", "avg_rank"],
    ascending=[True, True],
).reset_index(drop=True)

ordered_features_gbm = gbm_feature_ranking["feature"].tolist()

print("\nTop 15 features permitidas para GBM sin OTHER:")
display(gbm_feature_ranking.head(15).round(4))

# ------------------------------------------------------------
# 5) Entrenar top 7, 8, 9 y 10 en train; seleccionar por validacion; reportar test
# ------------------------------------------------------------

def compute_metrics(prefix, y_true, y_pred):
    return {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}_macro_f1": f1_score(y_true, y_pred, average="macro"),
        f"{prefix}_macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}_macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

gbm_topk_rows = []
models_by_k = {}

for k in TOP_K_VALUES:
    selected_features = ordered_features_gbm[:k]
    model = make_gbm_model()
    model.fit(X_train[selected_features], y_train)

    y_pred_valid = model.predict(X_valid[selected_features])
    y_pred_test = model.predict(X_test[selected_features])

    row = {
        "k_features": k,
        "features": selected_features,
    }
    row.update(compute_metrics("valid", y_valid, y_pred_valid))
    row.update(compute_metrics("test", y_test, y_pred_test))
    gbm_topk_rows.append(row)
    models_by_k[k] = model

gbm_topk_metrics = pd.DataFrame(gbm_topk_rows)

print("\nMetricas GBM por top-k, train=15%, valid=15%, test=70%, sin OTHER:")
display(gbm_topk_metrics.drop(columns=["features"]).round(4))

best_gbm_row = gbm_topk_metrics.sort_values(
    ["valid_macro_f1", "valid_balanced_accuracy", "k_features"],
    ascending=[False, False, True],
).iloc[0]
best_gbm_k = int(best_gbm_row["k_features"])
best_gbm_features = ordered_features_gbm[:best_gbm_k]
best_gbm_model = models_by_k[best_gbm_k]

class_labels = sorted(y.unique())
y_pred_test_best = best_gbm_model.predict(X_test[best_gbm_features])
cm = confusion_matrix(y_test, y_pred_test_best, labels=class_labels)

print("\nMejor seleccion por valid_macro_f1:")
print("k =", best_gbm_k)
print(best_gbm_features)
print("\nMetricas test del mejor GBM:")
for key, value in compute_metrics("test", y_test, y_pred_test_best).items():
    print(key + ":", round(value, 4))

# ------------------------------------------------------------
# 6) Guardar resultados
# ------------------------------------------------------------

gbm_dir = Path("gbm_no_other_results")
gbm_dir.mkdir(exist_ok=True)

gbm_ranking_csv = gbm_dir / "gbm_feature_ranking_no_other.csv"
gbm_topk_csv = gbm_dir / "gbm_topk_metrics_no_other.csv"
gbm_precheck_csv = gbm_dir / "gbm_feature_precheck_no_other.csv"
gbm_split_csv = gbm_dir / "gbm_split_15_15_70_no_other.csv"

gbm_feature_ranking.to_csv(gbm_ranking_csv, index=False)
gbm_topk_metrics.assign(features=gbm_topk_metrics["features"].apply(lambda values: ",".join(values))).to_csv(gbm_topk_csv, index=False)
feature_precheck_gbm.to_csv(gbm_precheck_csv, index=False)
split_summary.to_csv(gbm_split_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gbm_topk_metrics["k_features"], gbm_topk_metrics["valid_macro_f1"], marker="o", color=PRIMARY, label="Valid macro F1")
axes[0].plot(gbm_topk_metrics["k_features"], gbm_topk_metrics["test_macro_f1"], marker="s", color=SECONDARY, label="Test macro F1")
axes[0].plot(gbm_topk_metrics["k_features"], gbm_topk_metrics["test_balanced_accuracy"], marker="^", color=DARK, label="Test balanced acc")
axes[0].set_title("GBM sin OTHER: split 15/15/70")
axes[0].set_xlabel("Top-k features permitidas")
axes[0].set_ylabel("Score")
axes[0].grid(alpha=0.25)
axes[0].legend()

ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(
    ax=axes[1],
    cmap=OGLE_CMAP,
    colorbar=False,
    values_format="d",
)
axes[1].set_title(f"Test confusion matrix, GBM top {best_gbm_k}")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
gbm_summary_png = gbm_dir / "gbm_topk_metrics_confusion_no_other.png"
plt.savefig(gbm_summary_png, dpi=220, bbox_inches="tight")
plt.show()

selected_features_gbm_no_other = best_gbm_features
gbm_feature_ranking_no_other = gbm_feature_ranking
gbm_topk_metrics_no_other = gbm_topk_metrics
models_gbm_no_other = models_by_k
split_gbm_no_other = {
    "X_train": X_train,
    "X_valid": X_valid,
    "X_test": X_test,
    "y_train": y_train,
    "y_valid": y_valid,
    "y_test": y_test,
    "ids_test": ids_test,
    "types_test": types_test,
}

print("\nArchivos generados:")
print(gbm_ranking_csv)
print(gbm_topk_csv)
print(gbm_precheck_csv)
print(gbm_split_csv)
print(gbm_summary_png)


<!-- LABDOC_AUTO -->
### 8. Diagn?stico visual de fits Fourier

- **Teor?a:** La validaci?n visual de curvas plegadas detecta problemas que una m?trica agregada puede ocultar: alias, ruido, outliers o cobertura pobre de fase.
- **Llega:** Fotometr?a `phot/I` y features Fourier calculadas.
- **Sale:** Grillas de mejores, peores y ejemplos aleatorios en `fourier_fit_diagnostics/`, m?s `fourier_fit_examples_used.csv`.
- **Insights:** Las pulsantes suelen mostrar fits m?s coherentes; LPV y casos ruidosos/planos producen calidades menores, lo que hace ?til `fourier_fit_quality_I` como feature.


In [ ]:
# ============================================================
# Diagnostico visual de fits Fourier sobre curvas I plegadas
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)


try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR_CANDIDATES = [
    Path("OGLE3-GSEP-full"),
    Path("OGLE4-GSEP-full"),
]

data_dir = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if data_dir is None:
    raise FileNotFoundError("No encontre OGLE3-GSEP-full ni OGLE4-GSEP-full.")

phot_I_dir = data_dir / "phot" / "I"
features_path = Path("features_ogle_I_V_lombscargle_fourier.csv")

if not features_path.exists():
    raise FileNotFoundError(
        "No encontre features_ogle_I_V_lombscargle_fourier.csv. "
        "Ejecuta primero las celdas de Lomb-Scargle y Fourier."
    )

df_fourier_viz = pd.read_csv(features_path)

FOURIER_N_HARMONICS = 4
EXAMPLES_PER_CLASS = 3
MAX_POINTS_PER_CURVE = 800
RANDOM_STATE = 42

viz_dir = Path("fourier_fit_diagnostics")
viz_dir.mkdir(exist_ok=True)

CLASS_COLORS = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
}


def index_i_photometry_files(root_dir):
    files_by_id = {}

    for path in root_dir.rglob("*.dat"):
        rel_parts = path.relative_to(root_dir).parts[:-1]

        if "V" in rel_parts:
            continue

        object_id = path.stem
        previous = files_by_id.get(object_id)

        if previous is None or len(path.parts) < len(previous.parts):
            files_by_id[object_id] = path

    return files_by_id


def read_i_photometry(path):
    if path is None or not path.exists():
        return np.array([]), np.array([]), np.array([])

    try:
        data = np.loadtxt(path)
    except Exception:
        return np.array([]), np.array([]), np.array([])

    if data.ndim == 1:
        data = data.reshape(1, -1)

    if data.shape[1] < 2:
        return np.array([]), np.array([]), np.array([])

    time = data[:, 0].astype(float)
    mag = data[:, 1].astype(float)
    err = data[:, 2].astype(float) if data.shape[1] >= 3 else np.full_like(mag, np.nan)

    valid = np.isfinite(time) & np.isfinite(mag)
    return time[valid], mag[valid], err[valid]


def fourier_model_from_row(row, phase):
    model = np.full_like(phase, row["fourier_offset_I"], dtype=float)

    for harmonic in range(1, FOURIER_N_HARMONICS + 1):
        amp = row.get(f"fourier_A{harmonic}_I", np.nan)
        phi = row.get(f"fourier_phi{harmonic}_I", np.nan)

        if np.isfinite(amp) and np.isfinite(phi):
            model += amp * np.cos(2.0 * np.pi * harmonic * phase + phi)

    return model


def prepare_phase_curve(row, i_files):
    object_id = str(row["id"])
    period = row.get("fourier_period_I", np.nan)

    if not np.isfinite(period) or period <= 0:
        return None

    time, mag, err = read_i_photometry(i_files.get(object_id))

    if len(time) == 0:
        return None

    if len(time) > MAX_POINTS_PER_CURVE:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(len(time), size=MAX_POINTS_PER_CURVE, replace=False))
        time = time[keep]
        mag = mag[keep]
        err = err[keep]

    phase = ((time - np.nanmin(time)) / period) % 1.0
    order = np.argsort(phase)

    return phase[order], mag[order], err[order]


def plot_fourier_fit_grid(sample_df, title, out_path, i_files, n_cols=3):
    if sample_df.empty:
        print(f"No hay ejemplos para: {title}")
        return None

    n_rows = int(np.ceil(len(sample_df) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.1 * n_rows), squeeze=False)
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        prepared = prepare_phase_curve(row, i_files)

        if prepared is None:
            ax.axis("off")
            continue

        phase, mag, err = prepared
        phase_model = np.linspace(0, 2, 800)
        model = fourier_model_from_row(row, phase_model % 1.0)
        color = CLASS_COLORS.get(row.get("class_grouped", "OTHER"), MUTED)

        ax.scatter(phase, mag, s=9, alpha=0.45, color=color, label="I data")
        ax.scatter(phase + 1, mag, s=9, alpha=0.45, color=color)
        ax.plot(phase_model, model, color=DARK, linewidth=2.0, label="Fourier fit")
        ax.invert_yaxis()
        ax.set_xlim(0, 2)
        ax.set_xlabel("Fase")
        ax.set_ylabel("I mag")
        ax.grid(alpha=0.20)

        ax.set_title(
            f"{row['id']} | {row.get('class_grouped', '')}\n"
            f"P={row['fourier_period_I']:.4g} d | "
            f"R2~{row['fourier_var_ratio_I']:.2f} | "
            f"cov={row['fourier_phase_coverage_I']:.2f}",
            fontsize=10,
        )

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()
    return out_path


i_files = index_i_photometry_files(phot_I_dir)
print("Archivos I indexados:", len(i_files))

quality_cols = [
    "fourier_period_I",
    "fourier_var_ratio_I",
    "fourier_phase_coverage_I",
    "fourier_chi2_red_I",
    "fourier_resid_std_I",
]

quality_mask = (
    df_fourier_viz["fourier_period_I"].gt(0)
    & np.isfinite(df_fourier_viz["fourier_period_I"])
    & np.isfinite(df_fourier_viz["fourier_var_ratio_I"])
    & np.isfinite(df_fourier_viz["fourier_phase_coverage_I"])
)

df_quality = df_fourier_viz.loc[quality_mask].copy()

print("Objetos con fit Fourier valido:", len(df_quality), "de", len(df_fourier_viz))
print("\nResumen de calidad por clase:")
quality_summary = df_quality.groupby("class_grouped")[quality_cols].agg(["median", "mean", "count"])
display(quality_summary.round(4))

best_examples = (
    df_quality.sort_values(
        ["fourier_var_ratio_I", "fourier_phase_coverage_I"],
        ascending=[False, False],
    )
    .groupby("class_grouped", group_keys=False)
    .head(EXAMPLES_PER_CLASS)
)

worst_examples = (
    df_quality.sort_values(
        ["fourier_var_ratio_I", "fourier_phase_coverage_I"],
        ascending=[True, True],
    )
    .groupby("class_grouped", group_keys=False)
    .head(EXAMPLES_PER_CLASS)
)

rng = np.random.default_rng(RANDOM_STATE)
random_examples = (
    df_quality.groupby("class_grouped", group_keys=False)
    .apply(lambda part: part.sample(min(EXAMPLES_PER_CLASS, len(part)), random_state=RANDOM_STATE))
)

plot_fourier_fit_grid(
    best_examples,
    "Mejores fits Fourier por clase, curva I plegada",
    viz_dir / "fourier_best_fits_by_class.png",
    i_files,
)

plot_fourier_fit_grid(
    worst_examples,
    "Fits Fourier problematicos por clase, curva I plegada",
    viz_dir / "fourier_worst_fits_by_class.png",
    i_files,
)

plot_fourier_fit_grid(
    random_examples,
    "Muestra aleatoria de fits Fourier por clase, curva I plegada",
    viz_dir / "fourier_random_fits_by_class.png",
    i_files,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for class_name, group in df_quality.groupby("class_grouped"):
    color = CLASS_COLORS.get(class_name, MUTED)
    axes[0].scatter(
        np.log10(group["fourier_period_I"]),
        group["fourier_var_ratio_I"],
        s=14,
        alpha=0.45,
        label=class_name,
        color=color,
    )

axes[0].set_xlabel("log10(periodo Fourier I [dias])")
axes[0].set_ylabel("fourier_var_ratio_I")
axes[0].set_title("Calidad del fit vs periodo usado")
axes[0].grid(alpha=0.25)
axes[0].legend()

classes = sorted(df_quality["class_grouped"].dropna().unique())
box_data = [
    df_quality.loc[df_quality["class_grouped"] == cls, "fourier_var_ratio_I"].dropna().values
    for cls in classes
]
axes[1].boxplot(
    box_data,
    tick_labels=classes,
    showfliers=False,
    patch_artist=True,
    boxprops={"facecolor": LIGHT, "edgecolor": PRIMARY, "linewidth": 1.2},
    medianprops={"color": DARK, "linewidth": 2},
    whiskerprops={"color": MUTED},
    capprops={"color": MUTED},
)
axes[1].set_ylabel("fourier_var_ratio_I")
axes[1].set_title("Distribucion de calidad por clase")
axes[1].grid(axis="y", alpha=0.25)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
quality_png = viz_dir / "fourier_fit_quality_summary.png"
plt.savefig(quality_png, dpi=220, bbox_inches="tight")
plt.show()

examples_csv = viz_dir / "fourier_fit_examples_used.csv"
pd.concat([
    best_examples.assign(sample_type="best"),
    worst_examples.assign(sample_type="worst"),
    random_examples.assign(sample_type="random"),
]).to_csv(examples_csv, index=False)

print("\nArchivos generados:")
print(viz_dir / "fourier_best_fits_by_class.png")
print(viz_dir / "fourier_worst_fits_by_class.png")
print(viz_dir / "fourier_random_fits_by_class.png")
print(quality_png)
print(examples_csv)

print("\nLectura rapida:")
print("- fourier_var_ratio_I cerca de 1 indica que el modelo explica bien la varianza de la curva plegada.")
print("- Valores bajos o negativos indican periodo malo, curva no periodica clara, o Fourier insuficiente.")
print("- fourier_phase_coverage_I bajo indica huecos de fase; esos fits son menos confiables.")


<!-- LABDOC_AUTO -->
### 9. Dashboard final de comparaci?n GBM

- **Teor?a:** PCA ofrece una proyecci?n exploratoria del espacio de features, mientras ROC OVR y matrices de confusi?n eval?an clasificaci?n multiclase desde ?ngulos complementarios.
- **Llega:** Ranking permitido, features con calidad Fourier y partici?n 15/15/70 sin `OTHER`.
- **Sale:** Dashboard PCA/ROC, matrices top 7-10, ROC por clase y ejemplos de predicci?n en `gbm_final_dashboard/`.
- **Insights:** Top 8 mantiene el mejor balance validaci?n/test; los AUC son muy altos y los errores se concentran en fronteras donde las curvas comparten color/per?odo/forma.


In [ ]:
# ============================================================
# Dashboard final: GBM top 7-10 con split 15/15/70, PCA, ROC y predicciones
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta visual del laboratorio
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

PRIMARY = "#870047"
SECONDARY = "#ff6aa7"
DARK = "#3a0020"
LIGHT = "#ffc2da"
INK = "#1a1014"
MUTED = "#7a5060"

PLOT_PALETTE = [PRIMARY, SECONDARY, DARK, LIGHT, MUTED]
CLASS_COLORS_DEFAULT = {
    "LPV": PRIMARY,
    "BINARY": SECONDARY,
    "PULS": DARK,
    "OTHER": MUTED,
    "UNKNOWN": MUTED,
}
TYPE_COLORS = {
    "LPV": PRIMARY,
    "ECL": SECONDARY,
    "ELL": LIGHT,
    "RRLYR": DARK,
    "DSCT": PRIMARY,
    "DCEP": SECONDARY,
    "T2CEP": LIGHT,
    "ACEP": MUTED,
    "OTHER": MUTED,
}
OGLE_CMAP = LinearSegmentedColormap.from_list(
    "ogle_palette",
    [LIGHT, SECONDARY, PRIMARY, DARK],
)

plt.rcParams.update({
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "grid.color": LIGHT,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})
plt.rcParams["axes.prop_cycle"] = cycler(color=PLOT_PALETTE)


from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, label_binarize

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
TARGET_COL = "class_grouped"
EXCLUDED_CLASS = "OTHER"
TRAIN_SIZE = 0.15
VALID_SIZE = 0.15
TEST_SIZE = 0.70
TOP_K_VALUES = [7, 8, 9, 10]

source_path = Path("features_ogle_I_V_lombscargle_fourier_quality.csv")
ranking_path = Path("gbm_no_other_results/gbm_feature_ranking_no_other.csv")

if not source_path.exists():
    raise FileNotFoundError(
        "No encontre features_ogle_I_V_lombscargle_fourier_quality.csv. "
        "Ejecuta primero la celda GBM sin OTHER."
    )

if not ranking_path.exists():
    raise FileNotFoundError(
        "No encontre gbm_no_other_results/gbm_feature_ranking_no_other.csv. "
        "Ejecuta primero la celda GBM sin OTHER."
    )

def is_forbidden_mean_or_median_feature(feature):
    lower_name = feature.lower()
    return "mean" in lower_name or "median" in lower_name


def make_gbm_model():
    return HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        early_stopping=True,
        random_state=RANDOM_STATE,
    )


def reorder_proba_to_labels(model, proba, labels):
    ordered = np.zeros((proba.shape[0], len(labels)))

    for model_col, class_name in enumerate(model.classes_):
        label_col = labels.index(class_name)
        ordered[:, label_col] = proba[:, model_col]

    return ordered


def macro_roc_curve(y_binary, y_score, labels, n_grid=300):
    mean_fpr = np.linspace(0, 1, n_grid)
    interpolated_tprs = []
    class_auc_rows = []

    for class_idx, class_name in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_binary[:, class_idx], y_score[:, class_idx])
        class_auc = auc(fpr, tpr)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        interpolated_tprs.append(interp_tpr)
        class_auc_rows.append({"class": class_name, "roc_auc": class_auc})

    mean_tpr = np.mean(interpolated_tprs, axis=0)
    mean_tpr[-1] = 1.0
    macro_auc = auc(mean_fpr, mean_tpr)

    return mean_fpr, mean_tpr, macro_auc, pd.DataFrame(class_auc_rows)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


df_final_gbm = pd.read_csv(source_path)
feature_ranking_gbm = pd.read_csv(ranking_path)

df_final_gbm = df_final_gbm.loc[df_final_gbm[TARGET_COL] != EXCLUDED_CLASS].copy()
df_final_gbm[TARGET_COL] = df_final_gbm[TARGET_COL].astype(str)

ordered_features = [
    feature for feature in feature_ranking_gbm["feature"].tolist()
    if (
        feature in df_final_gbm.columns
        and pd.api.types.is_numeric_dtype(df_final_gbm[feature])
        and not is_forbidden_mean_or_median_feature(feature)
    )
]

if len(ordered_features) < max(TOP_K_VALUES):
    raise ValueError("El ranking no tiene suficientes features permitidas para comparar top 7-10.")

X_all = df_final_gbm[ordered_features].replace([np.inf, -np.inf], np.nan)
y_all = df_final_gbm[TARGET_COL]
ids_all = df_final_gbm["id"].astype(str) if "id" in df_final_gbm.columns else pd.Series(X_all.index.astype(str), index=X_all.index)
types_all = df_final_gbm["type"].astype(str) if "type" in df_final_gbm.columns else pd.Series("", index=X_all.index)

train_valid_size = TRAIN_SIZE + VALID_SIZE
valid_fraction_inside_train_valid = VALID_SIZE / train_valid_size

X_train_valid, X_test, y_train_valid, y_test, ids_train_valid, ids_test, types_train_valid, types_test = train_test_split(
    X_all,
    y_all,
    ids_all,
    types_all,
    train_size=train_valid_size,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid, ids_train, ids_valid, types_train, types_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    ids_train_valid,
    types_train_valid,
    test_size=valid_fraction_inside_train_valid,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "n_objects": [len(y_train), len(y_valid), len(y_test)],
    "fraction": [len(y_train) / len(y_all), len(y_valid) / len(y_all), len(y_test) / len(y_all)],
})

print("Split final usado:")
display(split_summary.round(4))
print("Features permitidas en el ranking top 10:")
print(ordered_features[:10])

class_labels = sorted(y_all.unique())
y_test_bin = label_binarize(y_test, classes=class_labels)

final_dashboard_dir = Path("gbm_final_dashboard")
final_dashboard_dir.mkdir(exist_ok=True)

CLASS_COLORS = {
    "BINARY": SECONDARY,
    "LPV": PRIMARY,
    "PULS": DARK,
}

model_results = {}
metrics_rows = []
roc_rows = []
all_prediction_rows = []

for k in TOP_K_VALUES:
    selected_features = ordered_features[:k]
    model = make_gbm_model()
    model.fit(X_train[selected_features], y_train)

    y_pred_valid = model.predict(X_valid[selected_features])
    y_pred_test = model.predict(X_test[selected_features])
    proba_test = reorder_proba_to_labels(
        model,
        model.predict_proba(X_test[selected_features]),
        class_labels,
    )

    macro_fpr, macro_tpr, macro_auc, class_auc_df = macro_roc_curve(y_test_bin, proba_test, class_labels)

    try:
        roc_auc_ovr_macro = roc_auc_score(
            y_test,
            proba_test,
            labels=class_labels,
            multi_class="ovr",
            average="macro",
        )
        roc_auc_ovr_weighted = roc_auc_score(
            y_test,
            proba_test,
            labels=class_labels,
            multi_class="ovr",
            average="weighted",
        )
    except ValueError:
        roc_auc_ovr_macro = macro_auc
        roc_auc_ovr_weighted = np.nan

    row = {"k_features": k, "features": selected_features}
    row.update({f"valid_{key}": value for key, value in compute_metrics(y_valid, y_pred_valid).items()})
    row.update({f"test_{key}": value for key, value in compute_metrics(y_test, y_pred_test).items()})
    row.update({
        "test_roc_auc_ovr_macro": roc_auc_ovr_macro,
        "test_roc_auc_ovr_weighted": roc_auc_ovr_weighted,
        "test_macro_roc_curve_auc": macro_auc,
    })
    metrics_rows.append(row)

    class_auc_df["k_features"] = k
    roc_rows.append(class_auc_df)

    pred_df = pd.DataFrame({
        "k_features": k,
        "id": ids_test.values,
        "type": types_test.values,
        "true_class": y_test.values,
        "pred_class": y_pred_test,
    }, index=X_test.index)

    for class_idx, class_name in enumerate(class_labels):
        pred_df[f"proba_{class_name}"] = proba_test[:, class_idx]

    pred_df["proba_pred"] = proba_test.max(axis=1)
    pred_df["proba_true"] = [
        proba_test[row_idx, class_labels.index(true_class)]
        for row_idx, true_class in enumerate(y_test.values)
    ]
    sorted_proba = np.sort(proba_test, axis=1)
    pred_df["confidence_margin"] = sorted_proba[:, -1] - sorted_proba[:, -2]
    pred_df["correct"] = pred_df["true_class"] == pred_df["pred_class"]
    all_prediction_rows.append(pred_df.reset_index(drop=True))

    model_results[k] = {
        "model": model,
        "features": selected_features,
        "y_pred_test": y_pred_test,
        "proba_test": proba_test,
        "macro_fpr": macro_fpr,
        "macro_tpr": macro_tpr,
        "macro_auc": macro_auc,
        "class_auc": class_auc_df,
        "confusion_matrix": confusion_matrix(y_test, y_pred_test, labels=class_labels),
    }

gbm_comparison_metrics = pd.DataFrame(metrics_rows)
gbm_roc_auc_by_class = pd.concat(roc_rows, ignore_index=True)
gbm_predictions_all_k = pd.concat(all_prediction_rows, ignore_index=True)

best_row = gbm_comparison_metrics.sort_values(
    ["valid_macro_f1", "valid_balanced_accuracy", "k_features"],
    ascending=[False, False, True],
).iloc[0]
best_k = int(best_row["k_features"])
best_features = model_results[best_k]["features"]

print("Metricas GBM top 7-10, train=15%, valid=15%, test=70%, sin OTHER:")
display(gbm_comparison_metrics.drop(columns=["features"]).round(4))

print("\nAUC ROC por clase en test:")
display(gbm_roc_auc_by_class.pivot(index="k_features", columns="class", values="roc_auc").round(4))

print("\nMejor modelo por valid_macro_f1:")
print("k =", best_k)
print(best_features)

# ------------------------------------------------------------
# PCA 2D con top 10 features permitidas
# ------------------------------------------------------------

pca_features = ordered_features[:10]
pca_imputer = SimpleImputer(strategy="median")
pca_scaler = RobustScaler()
pca = PCA(n_components=2, random_state=RANDOM_STATE)

X_train_pca = pca_scaler.fit_transform(pca_imputer.fit_transform(X_train[pca_features]))
pca.fit(X_train_pca)
X_all_pca_ready = pca_scaler.transform(pca_imputer.transform(X_all[pca_features]))
pca_coords = pca.transform(X_all_pca_ready)

pca_df = pd.DataFrame({
    "PC1": pca_coords[:, 0],
    "PC2": pca_coords[:, 1],
    "class_grouped": y_all.values,
    "id": ids_all.values,
    "type": types_all.values,
}, index=X_all.index)

best_pred_df = gbm_predictions_all_k.loc[gbm_predictions_all_k["k_features"] == best_k].copy()
misclassified_test_ids = set(best_pred_df.loc[~best_pred_df["correct"], "id"])
pca_df["misclassified_best_model"] = pca_df["id"].isin(misclassified_test_ids)

# ------------------------------------------------------------
# Figuras principales
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].plot(gbm_comparison_metrics["k_features"], gbm_comparison_metrics["valid_macro_f1"], marker="o", color=PRIMARY, label="Valid macro F1")
axes[0, 0].plot(gbm_comparison_metrics["k_features"], gbm_comparison_metrics["test_macro_f1"], marker="s", color=SECONDARY, label="Test macro F1")
axes[0, 0].plot(gbm_comparison_metrics["k_features"], gbm_comparison_metrics["test_roc_auc_ovr_macro"], marker="^", color=DARK, label="Test ROC AUC macro")
axes[0, 0].set_xlabel("Top-k features permitidas")
axes[0, 0].set_ylabel("Score")
axes[0, 0].set_title("Comparacion GBM con split 15/15/70")
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

for class_name, group in pca_df.groupby("class_grouped"):
    axes[0, 1].scatter(
        group["PC1"],
        group["PC2"],
        s=16,
        alpha=0.45,
        label=class_name,
        color=CLASS_COLORS.get(class_name, MUTED),
    )

misclassified = pca_df.loc[pca_df["misclassified_best_model"]]
axes[0, 1].scatter(
    misclassified["PC1"],
    misclassified["PC2"],
    s=45,
    facecolors="none",
    edgecolors=DARK,
    linewidths=1.2,
    label="Errores test",
)
pc1_limits = np.nanpercentile(pca_df["PC1"], [1, 99])
pc2_limits = np.nanpercentile(pca_df["PC2"], [1, 99])
pc1_pad = 0.08 * (pc1_limits[1] - pc1_limits[0])
pc2_pad = 0.08 * (pc2_limits[1] - pc2_limits[0])
axes[0, 1].set_xlim(pc1_limits[0] - pc1_pad, pc1_limits[1] + pc1_pad)
axes[0, 1].set_ylim(pc2_limits[0] - pc2_pad, pc2_limits[1] + pc2_pad)
axes[0, 1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% var train)")
axes[0, 1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% var train)")
axes[0, 1].set_title("PCA robusto top 10 features permitidas")
axes[0, 1].grid(alpha=0.20)
axes[0, 1].legend(markerscale=1.5)

for color, k in zip(PLOT_PALETTE, TOP_K_VALUES):
    axes[1, 0].plot(
        model_results[k]["macro_fpr"],
        model_results[k]["macro_tpr"],
        linewidth=2,
        color=color,
        label=f"Top {k}, AUC={model_results[k]['macro_auc']:.3f}",
    )
axes[1, 0].plot([0, 1], [0, 1], "--", color=MUTED, linewidth=1)
axes[1, 0].set_xlabel("False positive rate")
axes[1, 0].set_ylabel("True positive rate")
axes[1, 0].set_title("Curvas ROC macro promedio en test, OVR")
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend()

best_cm = model_results[best_k]["confusion_matrix"]
ConfusionMatrixDisplay(best_cm, display_labels=class_labels).plot(
    ax=axes[1, 1],
    cmap=OGLE_CMAP,
    colorbar=False,
    values_format="d",
)
axes[1, 1].set_title(f"Matriz de confusion test, mejor GBM top {best_k}")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
comparison_dashboard_png = final_dashboard_dir / "gbm_comparison_pca_roc_dashboard.png"
plt.savefig(comparison_dashboard_png, dpi=220, bbox_inches="tight")
plt.show()

# Matrices de confusion para k=7..10.
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, k in zip(axes, TOP_K_VALUES):
    ConfusionMatrixDisplay(
        model_results[k]["confusion_matrix"],
        display_labels=class_labels,
    ).plot(ax=ax, cmap=OGLE_CMAP, colorbar=False, values_format="d")
    ax.set_title(f"GBM top {k}")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
confusion_grid_png = final_dashboard_dir / "gbm_confusion_matrices_top7_10.png"
plt.savefig(confusion_grid_png, dpi=220, bbox_inches="tight")
plt.show()

# ROC por clase para el mejor modelo.
fig, ax = plt.subplots(figsize=(8, 6))
best_proba = model_results[best_k]["proba_test"]

for class_idx, class_name in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_test_bin[:, class_idx], best_proba[:, class_idx])
    class_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, color=CLASS_COLORS.get(class_name, MUTED), label=f"{class_name}, AUC={class_auc:.3f}")

ax.plot([0, 1], [0, 1], "--", color=MUTED, linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title(f"ROC por clase en test, mejor GBM top {best_k}")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
roc_by_class_png = final_dashboard_dir / "gbm_best_model_roc_by_class.png"
plt.savefig(roc_by_class_png, dpi=220, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# Ejemplos de predicciones en test
# ------------------------------------------------------------

best_predictions = best_pred_df.copy()
wrong_high_conf = (
    best_predictions.loc[~best_predictions["correct"]]
    .sort_values("proba_pred", ascending=False)
    .head(12)
    .assign(example_type="wrong_high_confidence")
)
correct_high_conf = (
    best_predictions.loc[best_predictions["correct"]]
    .sort_values("proba_pred", ascending=False)
    .head(12)
    .assign(example_type="correct_high_confidence")
)
low_confidence = (
    best_predictions
    .sort_values("confidence_margin", ascending=True)
    .head(12)
    .assign(example_type="low_confidence")
)

prediction_examples = pd.concat(
    [wrong_high_conf, low_confidence, correct_high_conf],
    ignore_index=True,
)

example_cols = [
    "example_type",
    "id",
    "type",
    "true_class",
    "pred_class",
    "correct",
    "proba_pred",
    "proba_true",
    "confidence_margin",
] + [f"proba_{class_name}" for class_name in class_labels]

print("\nEjemplos de predicciones del mejor modelo en test:")
display(prediction_examples[example_cols].round(4))

# ------------------------------------------------------------
# Guardar artefactos
# ------------------------------------------------------------

metrics_csv = final_dashboard_dir / "gbm_comparison_metrics_top7_10.csv"
roc_auc_csv = final_dashboard_dir / "gbm_roc_auc_by_class_top7_10.csv"
predictions_csv = final_dashboard_dir / "gbm_predictions_all_top7_10.csv"
examples_csv = final_dashboard_dir / "gbm_prediction_examples_best_model.csv"
pca_csv = final_dashboard_dir / "gbm_pca_coordinates_top10.csv"

metrics_to_save = gbm_comparison_metrics.copy()
metrics_to_save["features"] = metrics_to_save["features"].apply(lambda values: ",".join(values))
metrics_to_save.to_csv(metrics_csv, index=False)
gbm_roc_auc_by_class.to_csv(roc_auc_csv, index=False)
gbm_predictions_all_k.to_csv(predictions_csv, index=False)
prediction_examples[example_cols].to_csv(examples_csv, index=False)
pca_df.to_csv(pca_csv, index=False)

print("\nArchivos generados:")
print(metrics_csv)
print(roc_auc_csv)
print(predictions_csv)
print(examples_csv)
print(pca_csv)
print(comparison_dashboard_png)
print(confusion_grid_png)
print(roc_by_class_png)

# Variables reutilizables.
gbm_final_comparison_metrics = gbm_comparison_metrics
gbm_final_model_results = model_results
gbm_final_prediction_examples = prediction_examples
gbm_final_pca = pca_df
